# Day 061 — Exercise 4: Logged API

Wire `MetricsCollector` into a FastAPI app using `@app.middleware('http')`. Middleware runs **around every request** — before and after the route handler. This is the right place to measure latency and record status codes, because middleware sees both the request AND the final response.

```
Client → [middleware: start timer]
       → [route handler: build response]
       → [middleware: record status + duration]
       → Client
```

In [ ]:
# --- Provided: MetricsCollector (from Exercise 3) ---
import time

class MetricsCollector:
    def __init__(self):
        self._requests = 0
        self._errors   = 0
        self._latencies: list = []

    def record(self, status_code: int, duration_ms: float) -> None:
        self._requests += 1
        if status_code >= 400:
            self._errors += 1
        self._latencies.append(duration_ms)

    def summary(self) -> dict:
        avg        = sum(self._latencies) / len(self._latencies) if self._latencies else 0.0
        error_rate = self._errors / self._requests if self._requests else 0.0
        return {
            "requests":       self._requests,
            "errors":         self._errors,
            "avg_latency_ms": round(avg, 1),
            "error_rate":     round(error_rate, 3),
        }

    def reset(self) -> None:
        self._requests = 0
        self._errors   = 0
        self._latencies.clear()


In [ ]:
from fastapi import FastAPI, Request
from pydantic import BaseModel, Field
from starlette.testclient import TestClient


## Task

Implement `build_logged_api(process_fn=None) -> FastAPI`:

```
POST /ask   {"prompt": "..."}  → {"answer": str}  (422 if prompt empty)
GET /health                    → {"status": "ok"}
GET /metrics                   → collector.summary()
```

**Middleware** (`@app.middleware('http')`):
```python
async def _metrics(request: Request, call_next):
    start    = time.monotonic()
    response = await call_next(request)   # run the route
    duration = (time.monotonic() - start) * 1000
    collector.record(response.status_code, duration)
    return response
```

## Your Implementation

In [ ]:
def build_logged_api(process_fn=None) -> FastAPI:
    """FastAPI app with a metrics-recording middleware.

    POST /ask   {"prompt": str (min_length=1)} → {"answer": str}
    GET /health                                → {"status": "ok"}
    GET /metrics                               → MetricsCollector.summary()

    Middleware (registered with @app.middleware('http')):
        - Measure duration with time.monotonic()
        - After route runs: collector.record(response.status_code, duration_ms)

    process_fn: optional callable(prompt: str) -> str for testing.
    """
    # TODO: create app + collector, add middleware, add routes, return app
    raise NotImplementedError


In [ ]:
def build_logged_api(process_fn=None) -> FastAPI:
    app       = FastAPI()
    collector = MetricsCollector()

    @app.middleware("http")
    async def _metrics(request: Request, call_next):
        start    = time.monotonic()
        response = await call_next(request)
        duration = (time.monotonic() - start) * 1000
        collector.record(response.status_code, duration)
        return response

    class _AskReq(BaseModel):
        prompt: str = Field(min_length=1)

    @app.get("/health")
    def health():
        return {"status": "ok"}

    @app.get("/metrics")
    def metrics():
        return collector.summary()

    @app.post("/ask")
    def ask(req: _AskReq):
        answer = process_fn(req.prompt) if process_fn else req.prompt.upper()
        return {"answer": answer}

    return app


## Automated checks

In [ ]:
score, total = 0, 6
try:
    app    = build_logged_api(process_fn=str.upper)
    client = TestClient(app, raise_server_exceptions=False)

    # /health works
    r = client.get("/health")
    assert r.status_code == 200 and r.json()["status"] == "ok"
    score += 1; print("\u2705 GET /health returns 200 ok")

    # /ask returns answer
    r2 = client.post("/ask", json={"prompt": "hello"})
    assert r2.status_code == 200 and r2.json()["answer"] == "HELLO"
    score += 1; print("\u2705 POST /ask returns processed answer")

    # empty prompt → 422
    r3 = client.post("/ask", json={"prompt": ""})
    assert r3.status_code == 422
    score += 1; print("\u2705 empty prompt \u2192 422")

    # metrics recorded (3 requests so far; /metrics itself is 4th but not yet counted)
    rm = client.get("/metrics")
    assert rm.status_code == 200
    m = rm.json()
    assert m["requests"] == 3, f"Expected 3 requests, got {m['requests']}"
    score += 1; print("\u2705 /metrics shows 3 requests recorded")

    # errors >= 1 (the 422)
    assert m["errors"] >= 1, f"Expected errors >= 1, got {m['errors']}"
    score += 1; print("\u2705 /metrics shows at least 1 error (the 422)")

    # avg_latency_ms is a non-negative number
    assert isinstance(m["avg_latency_ms"], (int, float)) and m["avg_latency_ms"] >= 0
    assert 0 <= m["error_rate"] <= 1
    score += 1; print("\u2705 /metrics has valid avg_latency_ms and error_rate")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_logged_api(process_fn=None) -> FastAPI:
    app       = FastAPI()
    collector = MetricsCollector()

    @app.middleware("http")
    async def _metrics(request: Request, call_next):
        start    = time.monotonic()
        response = await call_next(request)
        duration = (time.monotonic() - start) * 1000
        collector.record(response.status_code, duration)
        return response

    class _AskReq(BaseModel):
        prompt: str = Field(min_length=1)

    @app.get("/health")
    def health():
        return {"status": "ok"}

    @app.get("/metrics")
    def metrics():
        return collector.summary()

    @app.post("/ask")
    def ask(req: _AskReq):
        answer = process_fn(req.prompt) if process_fn else req.prompt.upper()
        return {"answer": answer}

    return app
```

**Why does `/metrics` show 3 requests, not 4?** The middleware records AFTER `call_next` returns. When GET /metrics runs, the route handler calls `collector.summary()` — which sees the 3 previous requests. The middleware then records the /metrics request itself, but the response is already built. This off-by-one is expected and consistent.

</details>